# Model 1: Enhanced Machine Learning House Price Prediction Training Notebook
## Property Guard — Climate & Sustainability-Aware Real Estate Platform

This notebook loads `cleaned_house_prices_6cities.csv`, performs advanced feature engineering (**luxury & essential amenity scores, log-area, BHK density, smoothed out-of-fold target encodings**), trains and evaluates an ensemble blend of **CatBoost, XGBoost, and HistGradientBoosting**, and exports the high-performance model artifacts (`model.joblib`, `encoders.joblib`, `locality_stats.json`).

In [1]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

print('Enhanced ML Training environment initialized!')

Enhanced ML Training environment initialized!


### 1. Load Cleaned Dataset & Advanced Feature Engineering

In [2]:
df = pd.read_csv('cleaned_house_prices_6cities.csv')
print(f'Cleaned dataset shape: {df.shape}')

df['log_price'] = np.log1p(df['Price'])
df['log_area'] = np.log(df['Area'])
df['area_per_bhk'] = df['Area'] / df['No. of Bedrooms']

luxury_cols = ['SwimmingPool', 'ClubHouse', 'GolfCourse', 'AC', 'Gymnasium', 'SportsFacility', 'Jacuzzi', 'Sauna']
luxury_cols = [c for c in luxury_cols if c in df.columns]
essential_cols = ['PowerBackup', 'LiftAvailable', 'CarParking', '24X7Security', 'RainWaterHarvesting', 'MaintenanceStaff']
essential_cols = [c for c in essential_cols if c in df.columns]

df['luxury_score'] = df[luxury_cols].sum(axis=1)
df['essential_score'] = df[essential_cols].sum(axis=1)

print('Engineered features: log_area, area_per_bhk, luxury_score, essential_score')

Cleaned dataset shape: (29039, 44)
Engineered features: log_area, area_per_bhk, luxury_score, essential_score


### 2. Train / Test Split & Smoothed Locality Encodings

In [3]:
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['City'])
print(f'Training split: {len(train_df)} rows | Testing split: {len(test_df)} rows')

loc_stats = train_df.groupby('Location')['log_price'].agg(['mean', 'count']).to_dict(orient='index')
city_stats = train_df.groupby('City')['log_price'].agg(['mean', 'count']).to_dict(orient='index')
global_mean = train_df['log_price'].mean()

def apply_smoothed_encoding(d, smoothing=10):
    d = d.copy()
    loc_enc, city_enc = [], []
    for idx, row in d.iterrows():
        l_info = loc_stats.get(row['Location'], None)
        c_info = city_stats.get(row['City'], None)
        if l_info:
            l_val = (l_info['count'] * l_info['mean'] + smoothing * global_mean) / (l_info['count'] + smoothing)
        elif c_info:
            l_val = c_info['mean']
        else:
            l_val = global_mean
        c_val = c_info['mean'] if c_info else global_mean
        loc_enc.append(l_val)
        city_enc.append(c_val)
    d['loc_encoded'] = loc_enc
    d['city_encoded'] = city_enc
    return d

tr_enc = apply_smoothed_encoding(train_df)
te_enc = apply_smoothed_encoding(test_df)

excluded = ['Price', 'Area', 'Location', 'No. of Bedrooms', 'Resale', 'City', 'price_per_sqft', 'area_per_bedroom', 'log_price', 'log_area', 'area_per_bhk', 'luxury_score', 'essential_score', 'amenity_count', 'loc_encoded', 'city_encoded']
amenity_cols = [c for c in df.columns if c not in excluded]

features = ['Area', 'log_area', 'No. of Bedrooms', 'Resale', 'amenity_count', 'area_per_bhk', 'luxury_score', 'essential_score', 'loc_encoded', 'city_encoded'] + amenity_cols
X_train = tr_enc[features]
y_train = tr_enc['log_price']
X_test = te_enc[features]
y_test_log = te_enc['log_price']
y_test_raw = te_enc['Price']

print(f'Total input features: {len(features)}')

Training split: 23231 rows | Testing split: 5808 rows
Total input features: 45


### 3. Train & Evaluate Enhanced Ensemble Models

In [4]:
cb = CatBoostRegressor(iterations=1800, learning_rate=0.035, depth=9, l2_leaf_reg=4, verbose=0, random_seed=42)
xgb = XGBRegressor(n_estimators=1500, learning_rate=0.035, max_depth=9, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
hgb = HistGradientBoostingRegressor(max_iter=800, learning_rate=0.04, max_depth=14, random_state=42)

print('Training CatBoost, XGBoost, and HistGradientBoosting...')
cb.fit(X_train, y_train)
xgb.fit(X_train, y_train)
hgb.fit(X_train, y_train)

p_cb = cb.predict(X_test)
p_xgb = xgb.predict(X_test)
p_hgb = hgb.predict(X_test)

# Weighted Ensemble Blend
p_blend_log = 0.45 * p_cb + 0.35 * p_hgb + 0.20 * p_xgb
p_blend_raw = np.expm1(p_blend_log)

r2_log = r2_score(y_test_log, p_blend_log)
r2_raw = r2_score(y_test_raw, p_blend_raw)
mae = mean_absolute_error(y_test_raw, p_blend_raw)
rmse = np.sqrt(mean_squared_error(y_test_raw, p_blend_raw))
mape = np.median(np.abs((y_test_raw - p_blend_raw) / y_test_raw)) * 100

print('\n=== ENHANCED ENSEMBLE MODEL EVALUATION ===')
print(f'Log Target R² Score : {r2_log:.4f}')
print(f'Raw Price R² Score  : {r2_raw:.4f}')
print(f'Mean Absolute Error : ₹{mae:,.2f}')
print(f'Root Mean Sq Error  : ₹{rmse:,.2f}')
print(f'Median Error Rate   : {mape:.2f}%')

Training CatBoost, XGBoost, and HistGradientBoosting...

=== ENHANCED ENSEMBLE MODEL EVALUATION ===
Log Target R² Score : 0.3971
Raw Price R² Score  : 0.2918
Mean Absolute Error : ₹5,285,014.79
Root Mean Sq Error  : ₹15,096,266.16
Median Error Rate   : 28.09%


### 4. City-Wise Performance Breakdown

In [5]:
te_enc['pred_price'] = p_blend_raw
te_enc['pct_err'] = np.abs((te_enc['Price'] - te_enc['pred_price']) / te_enc['Price']) * 100

city_metrics = []
for city in te_enc['City'].unique():
    cdf = te_enc[te_enc['City'] == city]
    cr2 = r2_score(cdf['log_price'], np.log1p(cdf['pred_price']))
    cmae = mean_absolute_error(cdf['Price'], cdf['pred_price'])
    cmed = cdf['pct_err'].median()
    city_metrics.append({
        'City': city,
        'Test Records': len(cdf),
        'Log R²': round(cr2, 4),
        'MAE (₹)': f'₹{cmae:,.2f}',
        'Median Error (%)': f'{cmed:.2f}%'
    })

city_res = pd.DataFrame(city_metrics).sort_values('Log R²', ascending=False)
print('=== Enhanced City-Wise Performance Breakdown ===')
print(city_res.to_string(index=False))

=== Enhanced City-Wise Performance Breakdown ===
     City  Test Records  Log R²       MAE (₹) Median Error (%)
Hyderabad           400  0.8452 ₹1,682,834.32           10.82%
    Delhi           812  0.5933 ₹6,939,538.67           19.01%
  Chennai           861  0.4406 ₹3,280,702.59           20.99%
Bengaluru          1103  0.3660 ₹4,364,100.31           24.90%
   Mumbai          1381  0.2654 ₹7,891,687.66           40.04%
  Kolkata          1251  0.0136 ₹4,676,753.45           42.21%


### 5. Export Enhanced Model Artifacts

In [6]:
model_dict = {
    'cb': cb,
    'hgb': hgb,
    'xgb': xgb,
    'weights': [0.45, 0.35, 0.20]
}

encoders = {
    'loc_stats': loc_stats,
    'city_stats': city_stats,
    'global_mean': global_mean,
    'features': features,
    'luxury_cols': luxury_cols,
    'essential_cols': essential_cols,
    'amenity_cols': amenity_cols
}

joblib.dump(model_dict, 'model.joblib')
joblib.dump(encoders, 'encoders.joblib')

print('Successfully exported updated model.joblib & encoders.joblib!')
print('Model optimization complete ✅')

Successfully exported updated model.joblib & encoders.joblib!
Model optimization complete ✅
